# 02 응용: Constraint logit, attention mask와 평가 지표

논문의 internal constraint module이 만든 feasibility logit을 cross-attention과 최종 assignment gate에 사용하는 흐름을 작은 숫자로 재현한다.

In [1]:
from math import exp

def sigmoid(x): return 1/(1+exp(-x))
satellites = ['S1','S2','S3']
tasks = ['T1','T2','T3','T4']
constraint_logits = [
    [ 2.2, -1.0,  0.8, -2.0],
    [-0.5,  1.7,  1.1,  0.2],
    [ 0.4, -2.1,  2.5,  1.2],
]
assignment_scores = [
    [3.0, 4.5, 2.0, 5.0],
    [4.0, 3.2, 3.8, 2.0],
    [2.0, 5.0, 4.2, 3.9],
]
probabilities = [[round(sigmoid(x),3) for x in row] for row in constraint_logits]
probabilities

[[0.9, 0.269, 0.69, 0.119],
 [0.378, 0.846, 0.75, 0.55],
 [0.599, 0.109, 0.924, 0.769]]

In [2]:
def assign(threshold):
    result = {}
    for i, sat in enumerate(satellites):
        valid = [(assignment_scores[i][j], tasks[j], probabilities[i][j])
                 for j in range(len(tasks)) if probabilities[i][j] > threshold]
        result[sat] = max(valid)[1] if valid else 'NULL'
    return result

for tau in (0.3, 0.5, 0.7, 0.9):
    print(tau, assign(tau))

0.3 {'S1': 'T1', 'S2': 'T1', 'S3': 'T3'}
0.5 {'S1': 'T1', 'S2': 'T3', 'S3': 'T3'}
0.7 {'S1': 'T1', 'S2': 'T3', 'S3': 'T3'}
0.9 {'S1': 'NULL', 'S2': 'NULL', 'S3': 'T3'}


임계값이 낮으면 실행 불가능한 쌍이 통과할 위험이 있고, 높으면 가능한 작업도 모두 제거해 NULL이 많아진다. `τ_s`는 accuracy가 아니라 mission cost에 맞춰 calibration해야 한다.

In [3]:
# 논문의 여섯 지표와 종합 점수(CS)를 계산한다.
def metrics(required, progress, durations, completion_times_h, sensor_energy_wh):
    completed = [p >= r for p,r in zip(progress, required)]
    cr = 100*sum(completed)/len(required)
    pcr = 100*sum(min(p/r,1) for p,r in zip(progress,required))/len(required)
    wcr = 100*sum(d for d,c in zip(durations,completed) if c)/sum(durations)
    tat = sum(completion_times_h)/len(completion_times_h) if completion_times_h else 0
    pc = sum(sensor_energy_wh)
    # 식의 완수율 항은 0~1 비율로 해석한다.
    completion_term = .6*(cr/100)+.2*(pcr/100)+.2*(wcr/100)
    cs = (1/completion_term if completion_term else float('inf')) + tat/7 + pc/100
    return {'CR':cr,'PCR':pcr,'WCR':wcr,'TAT':tat,'PC':pc,'CS':cs}

result = metrics([10,20,15,8], [10,12,15,2], [10,20,15,8], [0.5,1.2], [3,5,4,2])
{k:round(v,3) for k,v in result.items()}

{'CR': 50.0, 'PCR': 71.25, 'WCR': 47.17, 'TAT': 0.85, 'PC': 14, 'CS': 2.124}

In [4]:
# 같은 완료 성능에서 전력만 줄이면 CS가 작아져야 한다.
efficient = metrics([10,20],[10,20],[10,20],[1,1],[2,2])
wasteful = metrics([10,20],[10,20],[10,20],[1,1],[20,20])
assert efficient['CS'] < wasteful['CS']
print('효율 모델 CS:', round(efficient['CS'],3), '고전력 모델 CS:', round(wasteful['CS'],3))

효율 모델 CS: 1.183 고전력 모델 CS: 1.543


## 실무 질문

- CR/PCR/WCR을 0~1과 0~100 중 어떤 scale로 식에 넣었는가?
- 임무가 재난 촬영이라면 전력 절감보다 missed critical task 비용이 더 크지 않은가?
- 위성별 threshold와 probability calibration이 필요한가?